# Chapter 13. 순환 신경망과 시계열 예측

## 1. 시간의 순서가 중요한 데이터

시계열은 시간 순서대로 기록된 데이터다. 매일의 기온, 시간대별 전기 사용량, 날짜별 철도 이용객 수가 여기에 해당한다. 숫자 자체뿐 아니라 언제 기록되었는지도 중요하다.

예를 들어 철도 이용객이 월요일부터 금요일까지 많고 주말에는 적다면, 날짜를 뒤섞었을 때 이 규칙을 알아보기 어려워진다. 시계열 예측은 과거의 흐름을 보고 아직 관측하지 않은 미래의 값을 추정하는 일이다.

PPT에서는 버스와 철도 이용객 데이터를 사용한다. 반복되는 요일 패턴을 살펴보고, 최근 여러 날의 기록을 이용해 다음 날이나 앞으로 여러 날의 이용객 수를 예측한다.

관련 슬라이드: 1, 7.


## 2. RNN은 이전 정보를 이어받는다

**순환 신경망**, 즉 RNN은 현재 입력과 이전 단계에서 전달받은 정보를 함께 사용한다. 앞의 내용을 조금 기억한 상태에서 다음 내용을 읽는 방식이라고 이해했다.

예를 들어 매일의 이용객 수를 차례로 읽는다면, 오늘 기록을 처리할 때 어제까지의 흐름을 요약한 정보도 함께 참고한다. 이 요약 정보를 **은닉 상태**라고 한다. 사람이 책을 읽으며 앞의 내용을 짧게 메모해 두는 것과 비슷하다.

현재 입력에 가중치를 적용하고, 이전 은닉 상태에도 가중치를 적용한다. 두 결과에 편향을 더하고 활성화 함수를 통과시키면 새로운 은닉 상태가 만들어진다. 가중치는 각 정보의 영향력을 정하는 숫자이고, 편향은 계산 기준을 조절하는 값이다.

PPT의 고리 모양 화살표는 같은 단계 안에서 끝없이 계산한다는 뜻이 아니다. 지금 만든 상태를 다음 시간 단계로 넘긴다는 뜻이다. 시간을 펼쳐 그리면 첫째 날, 둘째 날, 셋째 날로 상태가 이어진다.

이때 날짜마다 완전히 다른 계산 규칙을 만드는 것이 아니라 같은 가중치를 반복해서 사용한다. 다만 입력과 이전 상태가 달라지므로 매일 만들어지는 상태는 달라진다.

관련 슬라이드: 2.


## 3. 은닉 상태와 최종 예측값은 구분해야 한다

은닉 상태는 모델 내부에서 정보를 요약한 숫자 묶음이다. 사람이 읽을 수 있는 문장 형태의 메모는 아니며, 각 숫자가 ‘월요일 효과’처럼 명확한 이름 하나에 해당한다고 보장되지도 않는다.

이 은닉 상태를 이용해 최종 예측값을 만드는 층을 따로 둘 수 있다. PPT의 교통 예측 모델에서는 마지막 날까지 읽은 상태를 선형 층에 넣어 다음 날 이용객 수 하나를 만든다.

따라서 그림에서 다음 단계로 돌아가는 정보와 최종적으로 보여 주는 이용객 수는 구분해야 한다. 실제 예제는 예측한 이용객 수 하나를 매 시간 단계의 기억으로 넘기는 구조가 아니라, 은닉 상태를 전달하는 구조다.

관련 슬라이드: 2, 8.


## 4. 순전파는 입력으로 답을 만드는 과정이다

**순전파**는 입력을 모델에 넣어 예측값을 얻는 과정이다. RNN에서는 시간 순서대로 입력을 읽고 상태를 갱신한 뒤 필요한 예측을 만든다.

PPT의 기온 예시에서는 어제 기온 20도를 입력했을 때 오늘 기온을 25도로 예측한다. 실제 오늘 기온이 23도라면 예측이 2도 높다.

이 틀린 정도를 숫자로 나타낸 것이 **손실**이다. 오차를 제곱하면 2×2이므로 손실은 4가 된다. 맞혔는지 틀렸는지만 표시하는 것이 아니라 얼마나 틀렸는지 계산하는 것이다.

한 사례의 오차 제곱과 여러 사례의 평균 제곱 오차는 구분한다. 여러 날의 오차를 각각 제곱하고 평균하면 MSE가 된다.

관련 슬라이드: 3.


## 5. 역전파는 무엇을 고칠지 계산하는 과정이다

예측이 틀렸다는 사실만 알아서는 다음 예측이 좋아지지 않는다. 모델 안의 어떤 숫자를 어느 방향으로 바꿔야 손실이 줄어드는지 알아야 한다.

**역전파**는 손실에서 출발해 계산 과정을 거꾸로 따라가며 각 가중치의 영향을 구한다. ‘이 가중치를 조금 키우면 손실이 늘어날까, 줄어들까?’를 계산한다고 생각하면 된다. 이 변화의 방향과 정도를 나타내는 것이 **기울기**, 또는 gradient다.

역전파가 기울기를 구하면, 경사 하강법 같은 최적화 방법이 그 기울기를 이용해 가중치를 실제로 바꾼다. 역전파와 가중치 갱신은 연결되어 있지만 같은 작업은 아니다.

RNN에서는 현재 상태가 이전 상태에 연결되어 있으므로 시간 방향의 연결도 거슬러 올라가며 기울기를 구한다. 이를 **시간을 통한 역전파**, 즉 BPTT라고 한다. 같은 가중치가 여러 시점에서 사용되기 때문에 여러 시점에서 생긴 영향이 학습에 반영된다.

배우는 흐름은 입력 읽기, 예측 만들기, 손실 구하기, 기울기 계산하기, 가중치 바꾸기 순서로 이해했다.

관련 슬라이드: 3~4. BPTT는 시간 방향의 역전파를 이해하기 위한 보충 용어다.


## 6. 먼저 단순한 예측과 비교해야 한다

**베이스라인**은 새로운 방법이 얼마나 나은지 비교하기 위한 기본 방법이다. 복잡한 RNN을 만들었더라도 단순한 방법보다 잘 맞히는지 확인해야 의미가 있다.

**나이브 예측**은 과거 값을 그대로 가져와 미래 값으로 사용하는 단순한 방법이다. PPT에서는 주간 패턴을 고려해 일주일 전의 이용객 수를 사용한다. 다음 월요일을 예측할 때 지난 월요일의 값을 가져오는 식이다.

**계절성**은 일정한 주기로 패턴이 반복되는 성질이다. 여기서 계절은 봄·여름만 뜻하지 않는다. 요일에 따라 일주일마다 반복되는 패턴도 계절성에 포함된다.

**자기상관**은 같은 시계열에서 시간 간격을 두고 떨어진 값들이 얼마나 관련되어 있는지 나타낸다. 원래 기록과 7일 이동한 기록이 비슷하게 움직인다면 일주일 간격의 관련성이 있다는 뜻이다. 다만 관련성이 있다는 사실만으로 그 원인까지 알 수는 없다.

관련 슬라이드: 5, 7.


## 7. MAE, MAPE, MSE의 차이

**MAE는 평균 절대 오차**다. 예측값과 실제값의 차이에서 부호를 없앤 뒤 평균한다. 실제가 100명인데 110명으로 예측하거나, 90명으로 예측했다면 두 경우 모두 오차를 10명으로 센다. 단위가 원래 데이터와 같아 이용객 예측에서는 몇 명 정도 틀렸는지 이해하기 쉽다.

예를 들어 이틀의 실제 이용객이 각각 100명, 200명이고 예측이 110명, 180명이라면 오차의 크기는 10명과 20명이다. 평균을 내면 MAE는 15명이다.

**MAPE는 평균 절대 백분율 오차**다. 오차의 크기를 실제값으로 나누고 백분율로 나타낸 뒤 평균한다. 앞의 예에서는 10명 차이는 100명의 10%이고, 20명 차이는 200명의 10%이므로 MAPE는 10%다. 절대 인원보다 상대적으로 얼마나 틀렸는지 보는 방법이다.

MAPE는 실제값이 0이면 그대로 나눌 수 없고, 실제값이 매우 작으면 비율이 과도하게 커질 수 있다. 따라서 모든 데이터에 무조건 적합한 지표는 아니다.

**MSE는 평균 제곱 오차**다. 앞의 예에서 오차를 제곱하면 100과 400이고, 평균은 250이다. 큰 오차를 제곱하면 훨씬 커지므로 큰 실수에 더 큰 불이익을 준다. 이때 단위는 ‘명’이 아니라 ‘명의 제곱’이다.

어떤 지표를 쓸지는 문제에 따라 정한다. 같은 모델도 학습할 때는 MSE를 줄이고, 결과를 설명할 때는 해석하기 쉬운 MAE를 사용할 수 있다.

관련 슬라이드: 5.


## 8. ARMA는 과거 값과 과거 오차를 활용한다

시계열 예측에는 신경망 외에도 통계적인 방법이 있다. PPT에서는 ARMA, ARIMA, SARIMA를 소개한다.

**AR은 자기회귀**라는 뜻이다. 과거의 실제 값으로 현재나 미래 값을 설명한다. 예를 들어 어제의 이용객 수가 오늘의 이용객 수와 어떤 관계가 있는지 학습하는 방식이다.

**MA는 이동평균**이라는 이름을 쓰지만, 여기서는 최근 이용객 수 몇 개를 단순히 평균내는 방법과 다르다. 시계열 모델의 MA 부분은 과거의 예측 오차에 해당하는 정보를 활용한다. 이전에 예상보다 높거나 낮았던 정도를 다음 예측에 반영한다고 이해하면 된다.

**ARMA**는 이 두 아이디어를 결합한다. 과거 실제값의 흐름과 과거 오차의 영향을 함께 고려하는 모델이다.

관련 슬라이드: 6.


## 9. ARIMA와 SARIMA는 차분과 계절성을 더한다

시간이 지나며 평균 수준이나 변동의 성질이 계속 달라지는 시계열은 같은 규칙으로 설명하기 어려울 수 있다. 이런 성질과 관련해 **정상성**이라는 말을 사용한다. 쉽게 말해 시계열의 통계적인 성질이 시간에 따라 크게 달라지지 않는지를 보는 것이다.

**차분**은 현재 값에서 이전 값을 빼는 처리다. 이용객이 100명, 110명, 120명이라면 하루 전과의 차이는 10명, 10명이다. 전체 수준은 올라가지만 변화량으로 보면 일정한 패턴이 나타난다.

**ARIMA**는 ARMA에 차분을 결합한 모델이다. 모든 시계열에 반드시 차분을 적용하는 것은 아니고 필요한 정도를 설정한다. 차분만으로 모든 종류의 불안정한 변화를 해결하는 것도 아니다.

**SARIMA**는 일주일이나 일 년처럼 반복되는 계절성까지 고려한다. 일별 데이터에서 주기가 7이면 일주일 전과의 관계를 따로 반영할 수 있다.

PPT와 연결된 교재 코드에서는 일반 부분을 `(1, 0, 0)`, 계절 부분을 `(0, 1, 1, 7)`로 함께 지정한다. 일반 부분은 과거 값 한 단계, 일반 차분 없음, 일반 MA 항 없음이라는 뜻이다. 계절 부분은 계절 AR 항 없음, 계절 차분 한 번, 계절 MA 항 한 개, 주기 7을 뜻한다.

따라서 이 두 괄호는 완전히 별개의 모델 결과가 아니라 한 계절성 모델 안에 함께 들어가는 설정이다. 계절 차분 한 번은 이 경우 오늘 값에서 7일 전 값을 빼는 처리와 연결된다.

PPT에 나온 **그리드 탐색**은 여러 설정 조합을 시험하고 검증 결과를 비교하는 방법이다. 시험한 후보 중 좋은 설정을 고르는 것이며, 가능한 모든 상황에서 절대적으로 가장 좋은 값을 보장한다는 뜻은 아니다.

관련 슬라이드: 6.


## 10. 과거 56일을 입력으로 만들기

모델에 날짜별 기록을 그대로 한 줄씩 넣는 것만으로 학습 문제가 완성되는 것은 아니다. 무엇을 보고 무엇을 맞힐지 입력과 정답을 연결해야 한다.

PPT에서는 **과거 56일의 철도 이용객 수를 입력으로 주고, 57일째 이용객 수를 정답으로 둔다.** 56일은 8주이므로 여러 주의 반복 패턴을 담을 수 있다. 다만 56일이 모든 예측 문제에서 정답인 길이는 아니다.

첫 학습 사례가 1일째부터 56일째까지를 보고 57일째를 맞히는 것이라면, 다음 사례는 2일째부터 57일째까지를 보고 58일째를 맞힐 수 있다. 작은 창을 하루씩 옮기며 입력과 정답 묶음을 만드는 방식이다.

시계열에서는 평가 기준도 시간 순서를 따라야 한다. 과거 구간으로 학습하고 이후 구간을 예측하는 상황에 맞춰 검증해야 한다. 아직 알 수 없는 미래의 실제 이용객 수가 입력이나 전처리 기준에 들어가면, 실제 예측보다 성능이 좋게 보이는 데이터 누출이 생길 수 있다.

관련 슬라이드: 7. 시간 순서와 데이터 누출은 데이터 준비를 이해하기 위한 보충 설명이다.


## 11. 이용객 수를 100만으로 나누는 이유

교통 이용객 수는 수십만처럼 큰 숫자일 수 있다. PPT에서는 값을 100만으로 나눠 모델이 다루는 숫자 크기를 줄인다. 예를 들어 50만 명은 0.5가 된다.

이 처리를 **스케일링**이라고 한다. 데이터의 순서나 상대적인 크기를 유지하면서 숫자 범위를 바꾸는 것이다. 평균을 빼고 표준편차로 나누는 표준화와는 다른 처리다.

예측값이 0.48이라면 원래 단위로는 48만 명이다. MAE가 0.03이라면 원래 이용객 단위에서는 3만 명의 오차다. 결과를 설명할 때 원래 단위로 돌려놓는 일을 잊지 않아야 한다.

제곱 오차는 크기가 제곱으로 바뀌므로 MAE와 같은 방식으로만 환산하면 안 된다. 어떤 지표의 값을 보고 있는지도 함께 확인해야 한다.

관련 슬라이드: 7.


## 12. 입력 1개, 은닉 상태 32개, 출력 1개

PPT의 간단한 RNN은 입력 크기가 1이다. 이는 전체 입력에 숫자가 하나뿐이라는 뜻이 아니라, **하루마다 넣는 특성이 철도 이용객 수 하나**라는 뜻이다. 그런 하루 기록을 56일 순서로 넣는다.

은닉 상태 크기는 32다. 매일의 기록을 읽을 때 내부 요약 정보로 숫자 32개를 유지한다. 32일만 기억한다는 뜻도 아니고, 32개의 순환층을 쌓았다는 뜻도 아니다.

56일을 모두 읽은 뒤 마지막 은닉 상태를 선형 층에 전달한다. 선형 층은 32개의 요약 숫자를 조합해 다음 날 이용객 수 하나를 만든다. 그래서 출력 크기는 1이다.

원본 코드에서 마지막 시점의 출력을 선택하는 부분은 ‘56일 중 마지막 날까지 반영된 상태를 사용한다’는 뜻이다. 마지막 하루의 원자료만 사용하는 것은 아니다.

이 모델은 여러 시점의 입력으로 하나의 예측을 만드는 구조로 이해했다.

관련 슬라이드: 8.


## 13. PPT의 예측 성능 비교 읽기

PPT에서는 MAE를 이용객 수 단위로 비교한다. 일주일 전 값을 가져온 나이브 예측은 약 42,143명, 선형 모델은 약 37,726명, SARIMA는 약 32,041명, 간단한 RNN은 약 30,659명의 오차로 제시되어 있다.

이 비교에서는 MAE가 가장 작은 간단한 RNN이 제시된 네 방법 중 가장 좋은 결과다. MAE가 작다는 것은 날짜별 예측과 실제 이용객 수의 차이가 평균적으로 작다는 뜻이다.

선형 모델은 과거 입력값들을 가중합해 예측하는 방법으로 이해할 수 있다. RNN은 순서대로 상태를 갱신하며 과거 흐름을 요약한다. 통계 모델과 단순 모델을 함께 비교하면 RNN이 얼마나 도움이 되는지 판단하기 쉽다.

다만 이 숫자로 RNN이 모든 시계열에서 항상 가장 좋다고 결론 내릴 수는 없다. 데이터 구간, 전처리, 학습 설정, 평가 방식에 따라 결과는 달라진다. MAE가 30,659명이라고 해서 모든 날짜에 정확히 그만큼 틀렸다는 뜻도 아니다.

여기 적은 값은 PPT에 제시된 교재 기준 결과이며, 이 노트북에서 재현 실험을 수행한 결과가 아니다.

관련 슬라이드: 9.


## 14. 한 번에 여러 날을 예측하기

다음 날 하나만이 아니라 앞으로 14일을 알고 싶을 수도 있다. 이를 **다단계 예측**이라고 한다.

한 가지 방법은 다음 날을 예측하고, 그 예측값을 다시 입력에 넣어 그다음 날을 예측하는 것이다. 하지만 첫날 예측이 틀렸다면 그 틀린 값이 다음 예측의 입력이 된다. 이런 과정에서 오차가 이어져 커질 수 있다.

다른 방법은 한 번에 앞으로 14일의 값을 출력하는 것이다. 이전 예측을 다시 입력해야 하는 과정을 줄일 수 있다. 다만 여러 날을 직접 예측한다고 해서 예측 오차 자체가 없어지는 것은 아니다.

PPT에서 설명하는 **Sequence-to-Sequence** 방식은 각 입력 시점마다 그 시점 이후 14일의 값을 출력한다. 과거 기록이 차례로 들어올 때마다 미래 14일을 예측하는 숫자 묶음을 만드는 구조다. 마지막 시점의 출력은 가장 최근 입력을 기준으로 한 14일 예측이 된다.

학습할 때 여러 시점의 출력에 손실을 줄 수 있으므로 마지막 출력 하나만 학습하는 것보다 많은 학습 신호를 얻을 수 있다. 그렇다고 학습 시간이나 안정성이 언제나 좋아지는 것은 아니며, 실제 결과로 비교해야 한다.

관련 슬라이드: 10.


## 15. 철도뿐 아니라 버스와 요일도 함께 보기

하루마다 철도 이용객 수 하나만 넣으면 **단변량 시계열**이다. 철도 이용객과 버스 이용객 등 여러 값을 함께 넣으면 **다변량 시계열**이다. 시간의 길이와 별개로 각 시점에서 제공하는 정보의 종류가 늘어나는 것이다.

버스와 철도 이용객은 함께 출퇴근 패턴의 영향을 받을 수 있다. 두 자료를 함께 보면 철도 기록 하나만으로는 파악하기 어려운 관계를 학습할 가능성이 있다. 다만 정보가 늘었다고 정확도가 반드시 높아지는 것은 아니다.

요일 유형 같은 범주형 정보도 넣을 수 있다. **원-핫 인코딩**은 종류마다 칸을 만들고 해당하는 칸만 1, 나머지는 0으로 표시하는 방법이다. 세 범주가 있다면 각각 `(1, 0, 0)`, `(0, 1, 0)`, `(0, 0, 1)`처럼 나타낸다. 이렇게 하면 요일 유형에 임의의 크기 순서를 부여하지 않는다.

PPT에서는 평일·주말·휴일 같은 유형을 예로 든다. 실제 구현에서는 데이터에 정의된 범주를 따라야 한다. 연결된 교재 예제는 다음 날의 날짜 유형처럼 미리 알 수 있는 정보도 활용한다.

내일이 휴일이라는 달력 정보는 예측 시점에도 알 수 있지만, 내일 실제 버스 이용객 수는 아직 알 수 없다. 미래 정보라고 모두 같은 것이 아니라 **예측할 때 실제로 이용 가능한 정보인지**를 구분해야 한다.

관련 슬라이드: 10.


## 16. 심층 RNN은 순환층을 여러 겹 쌓는다

**심층 RNN**은 순환층을 여러 개 쌓은 모델이다. 첫 번째 층이 시간 흐름에서 특징을 찾으면 다음 층은 그 특징의 흐름을 다시 처리한다. 아래층의 요약을 위층이 한 번 더 해석하는 모습으로 이해했다.

여기서 시간 방향으로 56일을 처리하는 것과 순환층을 여러 겹 쌓는 것은 다르다. 전자는 입력의 길이이고, 후자는 모델의 깊이다.

층을 늘리면 더 복잡한 패턴을 표현할 수 있지만 계산량도 늘고 학습이 어려워질 수 있다. 층을 깊게 만든다는 이유만으로 오래전 정보를 잘 기억하게 되는 것은 아니다.

오래된 정보가 여러 시간 단계를 지나며 약해지는 문제는 별도로 생각해야 한다. 이 문제를 줄이도록 기억의 흐름을 조절하는 대표적인 구조가 LSTM과 GRU다.

관련 슬라이드: 11~12.


## 17. LSTM은 기억을 남기고 바꾸는 양을 조절한다

기본 RNN에서는 오래전 정보가 여러 단계를 거치며 약해질 수 있다. **LSTM**은 기억을 유지하고 바꾸는 통로를 더 세밀하게 설계한 순환 구조다.

LSTM은 **은닉 상태 h**와 **셀 상태 c**를 전달한다. 은닉 상태는 현재 단계에서 다음 계산이나 출력에 활용하는 정보이고, 셀 상태는 정보를 시간에 걸쳐 유지하도록 만든 별도의 통로다. 짧게 보여 주는 요약과 계속 고쳐 쓰는 장기 메모에 비유할 수 있다. 다만 두 상태의 역할이 ‘짧은 기억’과 ‘긴 기억’으로 완전히 나뉜다고 단정할 수는 없다.

**망각 게이트**는 이전 셀 상태를 얼마나 남길지 조절한다. **입력 게이트**는 새로 만든 후보 정보를 셀 상태에 얼마나 추가할지 조절한다. **출력 게이트**는 갱신된 셀 상태에서 어느 정도를 은닉 상태로 내보낼지 조절한다.

게이트는 딱 열리거나 닫히기만 하는 스위치가 아니다. 주로 0과 1 사이의 값을 이용해 정보가 통과하는 양을 조절하는 수도꼭지처럼 생각하면 된다. 이 조절 방법도 학습을 통해 정해진다.

세 게이트가 서로 다른 역할을 맡지만, 서로 아무 관계 없이 움직인다는 뜻은 아니다. 현재 입력과 이전 상태를 바탕으로 함께 작동하며 기억을 갱신한다. 오래된 정보를 유지하기 쉬운 경로를 만든다는 것이 핵심이다.

관련 슬라이드: 12.


## 18. GRU는 기억 구조를 간단하게 만든다

**GRU**도 과거 정보를 얼마나 유지하고 새 정보를 얼마나 반영할지 조절하는 순환 구조다. LSTM처럼 별도의 셀 상태를 전달하지 않고 은닉 상태 하나를 중심으로 처리한다.

**업데이트 게이트**는 이전 상태와 새로 만든 후보 상태를 어느 정도 섞을지 조절한다. **리셋 게이트**는 새로운 후보 상태를 계산할 때 이전 상태를 얼마나 참고할지 조절한다.

리셋이라는 이름 때문에 기억을 무조건 전부 지우는 동작으로 이해하면 안 된다. 이전 정보의 영향을 줄이거나 유지하도록 학습하는 조절 장치다. 업데이트 역시 비워 둔 칸에 정보를 채우는 식의 문자 그대로의 저장 과정은 아니다.

GRU에는 LSTM과 같은 별도의 출력 게이트가 없다. 같은 입력 크기와 은닉 크기를 기준으로 구조가 간단하고 파라미터가 적어 학습이나 계산이 더 가벼울 수 있다. 실제 속도와 성능은 데이터와 구현 환경에 따라 달라진다.

LSTM이 항상 더 정확하거나 GRU가 항상 더 좋은 것은 아니다. 오래된 정보를 유지하도록 설계했다는 공통점과, 상태와 게이트를 어떻게 구성하는지의 차이를 이해한 뒤 같은 조건에서 비교해야 한다.

관련 슬라이드: 12.
